# 00. Оркестратор конвеєра курсів валют НБУ

Цей ноутбук відповідає за запуск усіх етапів конвеєра у правильному порядку. Сам оркестратор не завантажує і не перетворює курси валют. Його завдання полягає в тому, щоб запускати інші ноутбуки у правильному порядку, зупиняти процес у разі помилки та зберігати журнал запуску.

Кожен наступний ноутбук використовує результат попереднього. Тому, якщо один з етапів завершився помилкою, виконувати наступні етапи немає сенсу.

Виконані копії ноутбуків разом із виводами комірок зберігаються в папці `runs`. Інформація про статус і тривалість кожного етапу записується в таблицю `nbu_meta.run_log`.

## Послідовність роботи

1. Підключитися до BigQuery.
2. Встановити та імпортувати `papermill`.
3. Задати порядок виконання ноутбуків.
4. Запустити ноутбуки по черзі.
5. Зупинити конвеєр після першої помилки.
6. Зібрати журнал виконання в DataFrame.
7. Дописати журнал у BigQuery.
8. Вивести підсумок прогону та перевірити результат.

## 1. Налаштування та підключення

Вказуємо назву Google Cloud проєкту та регіон BigQuery. Також задаємо назви датасету і таблиці, у яких буде зберігатися технічний журнал запусків.

Оркестратор не працює безпосередньо з таблицями курсів валют. Він тільки запускає робочі ноутбуки та записує інформацію про їх виконання.

In [1]:
PROJECT_ID = "nbu-bigquery-etl"
LOCATION = "EU"

META_DATASET = f"{PROJECT_ID}.nbu_meta"
RUN_LOG_TABLE = f"{META_DATASET}.run_log"

In [2]:
import os
import sys
import json
import uuid

import pandas as pd
from google.cloud import bigquery

pd.set_option("display.max_columns", 40)

In [3]:
creds = None

if "google.colab" in sys.modules:
    from google.colab import auth
    auth.authenticate_user()

elif os.environ.get("GCP_SA_KEY"):
    from google.oauth2 import service_account

    creds = service_account.Credentials.from_service_account_info(
        json.loads(os.environ["GCP_SA_KEY"]),
        scopes=["https://www.googleapis.com/auth/cloud-platform"]

    )

client = bigquery.Client(project=PROJECT_ID, location=LOCATION, credentials=creds)

print("проєкт:", client.project)

проєкт: nbu-bigquery-etl


## Завдання 7. Ноутбук 00: оркестратор

### Завдання 7.1. Встановлення papermill

`papermill` дозволяє програмно запускати Jupyter-ноутбуки. Для кожного ноутбука він створює окреме ядро та зберігає виконану копію разом із виводами комірок.

Використовуємо `%pip`, щоб установити пакет у середовище, в якому працює цей ноутбук.

In [4]:
%pip install papermill

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Завдання 7.2. Порядок виконання ноутбуків

Записуємо назви робочих ноутбуків у тому порядку, у якому вони повинні виконуватися.

Порядок важливий, тому що кожен наступний етап читає дані, створені попередніми етапами. Наприклад, таблицю фактів не можна побудувати раніше за вимір валют.

Також створюємо папку `runs`. У ній `papermill` зберігатиме виконані копії ноутбуків. Параметр `exist_ok=True` дозволяє повторно запускати цю комірку, навіть якщо папка вже існує.

In [5]:
NOTEBOOKS = [
    "01_bronze.ipynb",
    "02_dim_currency.ipynb",
    "03_dim_date.ipynb",
    "04_snapshot.ipynb",
    "05_fact_merge.ipynb",
]

os.makedirs("runs", exist_ok=True)

Перед запуском перевіряємо поточну робочу папку та наявність усіх файлів. Відносні шляхи працюють тільки тоді, коли Python шукає файли у правильній папці.

Якщо навпроти кожного ноутбука отримано `True`, оркестратор зможе передати його в `papermill`.

In [6]:
print("Поточна робоча папка:", os.getcwd())

for name in NOTEBOOKS:
    print(name, "знайдено:", os.path.exists(name))

Поточна робоча папка: e:\Data Engineering\DataLab\Data Lake\practise_10
01_bronze.ipynb знайдено: True
02_dim_currency.ipynb знайдено: True
03_dim_date.ipynb знайдено: True
04_snapshot.ipynb знайдено: True
05_fact_merge.ipynb знайдено: True


### Завдання 7.3 та 7.4. Запуск ноутбуків і зупинка після помилки

Створюємо один `run_id` для всього прогону та запускаємо ноутбуки по черзі.

Для кожного етапу окремо запам’ятовуємо час початку і завершення. Якщо ноутбук виконався успішно, записуємо статус `OK`. Якщо виникла помилка, записуємо статус `FAILED`, зберігаємо текст помилки та зупиняємо цикл.

Зупинка потрібна тому, що наступні ноутбуки залежать від результату попереднього етапу.

In [7]:
import papermill as pm

run_id = str(uuid.uuid4())
log_rows = []

for step_no, name in enumerate(NOTEBOOKS, start=1):
    started = pd.Timestamp.now(tz="UTC")
    try:
        pm.execute_notebook(name, f"runs/{name}", kernel_name="python3")
        status, error = "OK", None
    except Exception as e:
        status, error = "FAILED", f"{type(e).__name__}: {e}"[:1000]

    finished = pd.Timestamp.now(tz="UTC")
    duration_sec = (finished - started).total_seconds()

    log_rows.append({"run_id": run_id,
                     "step_no": step_no,
                     "notebook": name,
                     "status": status,
                     "started_at": started,
                     "finished_at": finished,
                     "duration_sec": duration_sec,
                     "error": error})

    print(f"{step_no}. {name}: {status}, "
    f"тривалість {duration_sec:.2f} сек.")

    if status == "FAILED":
        print(f"Конвеєр зупинено на ноутбуці: {name}")
        break

e:\Data Engineering\DataLab\Python\dataeng\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Executing: 100%|██████████| 25/25 [00:09<00:00,  2.68cell/s]


1. 01_bronze.ipynb: OK, тривалість 9.38 сек.


Executing: 100%|██████████| 25/25 [00:07<00:00,  3.13cell/s]


2. 02_dim_currency.ipynb: OK, тривалість 7.99 сек.


Executing: 100%|██████████| 21/21 [00:08<00:00,  2.48cell/s]


3. 03_dim_date.ipynb: OK, тривалість 8.46 сек.


Executing: 100%|██████████| 24/24 [00:11<00:00,  2.06cell/s]


4. 04_snapshot.ipynb: OK, тривалість 11.68 сек.


Executing: 100%|██████████| 27/27 [00:10<00:00,  2.46cell/s]

5. 05_fact_merge.ipynb: OK, тривалість 10.97 сек.


### Завдання 7.5. Формування журналу прогону

Під час виконання циклу результат кожного етапу додавався до списку `log_rows` у вигляді словника.

Тепер перетворюємо список словників на DataFrame. Один рядок DataFrame відповідає одному виконаному ноутбуку, а колонки містять його номер, статус, час виконання та можливу помилку.

Усі рядки одного прогону мають однаковий `run_id`. Це дозволяє знайти в журналі всі етапи, які належать до одного запуску оркестратора.

In [8]:
run_log_df = pd.DataFrame(log_rows)

run_log_df

,run_id,step_no,notebook,status,started_at,finished_at,duration_sec,error
0,0fae47f8-75ee-4f92-a312-58df50186b7e,1,01_bronze.ipynb,OK,2026-08-24 12:05:35.439990+00:00,2026-08-24 12:05:44.816942+00:00,9.376952,None
1,0fae47f8-75ee-4f92-a312-58df50186b7e,2,02_dim_currency.ipynb,OK,2026-08-24 12:05:44.817140+00:00,2026-08-24 12:05:52.805754+00:00,7.988614,None
2,0fae47f8-75ee-4f92-a312-58df50186b7e,3,03_dim_date.ipynb,OK,2026-08-24 12:05:52.805952+00:00,2026-08-24 12:06:01.265580+00:00,8.459628,None
3,0fae47f8-75ee-4f92-a312-58df50186b7e,4,04_snapshot.ipynb,OK,2026-08-24 12:06:01.265779+00:00,2026-08-24 12:06:12.950182+00:00,11.684403,None
4,0fae47f8-75ee-4f92-a312-58df50186b7e,5,05_fact_merge.ipynb,OK,2026-08-24 12:06:12.950279+00:00,2026-08-24 12:06:23.921652+00:00,10.971373,None


### Завдання 7.6. Збереження журналу в BigQuery

Створюємо окремий датасет `nbu_meta` для технічної інформації про роботу конвеєра. Тут зберігатимуться не курси валют, а результати запуску ноутбуків.

Для таблиці `run_log` задаємо схему вручну. Це дозволяє заздалегідь визначити тип кожної колонки та не покладатися на автоматичне розпізнавання BigQuery.

Таблицю партиціюємо за полем `started_at`. Завдяки цьому записи різних дат зберігатимуться в окремих частинах таблиці.

Новий журнал додаємо через `WRITE_APPEND`, щоб не видаляти інформацію про попередні запуски конвеєра.

In [9]:
dataset = bigquery.Dataset(META_DATASET)
dataset.location = LOCATION

client.create_dataset(dataset, exists_ok=True)

print("датасет:", META_DATASET)

датасет: nbu-bigquery-etl.nbu_meta


In [10]:
schema = [
    bigquery.SchemaField("run_id",       "STRING",    mode="REQUIRED"),
    bigquery.SchemaField("step_no",      "INTEGER",   mode="REQUIRED"),
    bigquery.SchemaField("notebook",     "STRING",    mode="REQUIRED"),
    bigquery.SchemaField("status",       "STRING",    mode="REQUIRED"),
    bigquery.SchemaField("started_at",   "TIMESTAMP", mode="REQUIRED"),
    bigquery.SchemaField("finished_at",  "TIMESTAMP", mode="REQUIRED"),
    bigquery.SchemaField("duration_sec", "FLOAT",     mode="REQUIRED"),
    bigquery.SchemaField("error",        "STRING",    mode="NULLABLE"),
]

table = bigquery.Table(RUN_LOG_TABLE, schema=schema)

table.time_partitioning = bigquery.TimePartitioning(field="started_at")

client.create_table(table, exists_ok=True)
print("таблиця готова:", RUN_LOG_TABLE)

таблиця готова: nbu-bigquery-etl.nbu_meta.run_log


In [11]:
cfg = bigquery.LoadJobConfig(schema=schema, write_disposition="WRITE_APPEND")

load_job = client.load_table_from_dataframe(run_log_df, RUN_LOG_TABLE, job_config=cfg)

load_job.result()

table = client.get_table(RUN_LOG_TABLE)

print("таблиця:", RUN_LOG_TABLE)
print("рядків у поточному прогоні:", len(run_log_df))
print("усього рядків у таблиці:", table.num_rows)

e:\Data Engineering\DataLab\Python\dataeng\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


таблиця: nbu-bigquery-etl.nbu_meta.run_log
рядків у поточному прогоні: 5
усього рядків у таблиці: 13


### Завдання 7.7. Підсумок прогону та перевірка журналу

Після завершення конвеєра рахуємо кількість успішних і невдалих етапів, а також їхню загальну тривалість.

Підсумок поточного прогону обчислюємо з DataFrame `run_log_df`. Потім виконуємо SQL-запит до `nbu_meta.run_log`, щоб переконатися, що журнал був збережений у BigQuery.

DataFrame показує результат поточного прогону, а таблиця BigQuery містить історію всіх прогонів.

In [12]:
ok_count = (run_log_df["status"] == "OK").sum()
failed_count = (run_log_df["status"] == "FAILED").sum()
total_duration_sec = run_log_df["duration_sec"].sum()

print("Підсумок прогону")
print("run_id:", run_id)
print("успішно:", ok_count)
print("з помилкою:", failed_count)
print(f"загальна тривалість: {total_duration_sec:.2f} сек.")

Підсумок прогону
run_id: 0fae47f8-75ee-4f92-a312-58df50186b7e
успішно: 5
з помилкою: 0
загальна тривалість: 48.48 сек.


In [13]:
check_sql = f"""
SELECT run_id,
       step_no,
       notebook,
       status,
       duration_sec
  FROM `{RUN_LOG_TABLE}`
ORDER BY started_at DESC
LIMIT 10
"""

check_df = client.query(check_sql).to_dataframe()

check_df

e:\Data Engineering\DataLab\Python\dataeng\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,run_id,step_no,notebook,status,duration_sec
0,0fae47f8-75ee-4f92-a312-58df50186b7e,5,05_fact_merge.ipynb,OK,10.971373
1,0fae47f8-75ee-4f92-a312-58df50186b7e,4,04_snapshot.ipynb,OK,11.684403
2,0fae47f8-75ee-4f92-a312-58df50186b7e,3,03_dim_date.ipynb,OK,8.459628
3,0fae47f8-75ee-4f92-a312-58df50186b7e,2,02_dim_currency.ipynb,OK,7.988614
4,0fae47f8-75ee-4f92-a312-58df50186b7e,1,01_bronze.ipynb,OK,9.376952
5,57225d1d-ca77-4065-83a6-c66278c28d81,3,03_dim_date.ipynb,FAILED,1.497923
6,57225d1d-ca77-4065-83a6-c66278c28d81,2,02_dim_currency.ipynb,OK,6.956871
7,57225d1d-ca77-4065-83a6-c66278c28d81,1,01_bronze.ipynb,OK,10.031334
8,7bb036f3-299c-474b-93de-30324718ecb8,5,05_fact_merge.ipynb,OK,10.693059
9,7bb036f3-299c-474b-93de-30324718ecb8,4,04_snapshot.ipynb,OK,8.458020


### Завдання 7.8. Перевірка роботи оркестратора

Спочатку запускаємо оркестратор із робочими ноутбуками. Очікуємо, що всі п’ять етапів отримають статус `OK`.

Потім тимчасово створюємо помилку в одному з робочих ноутбуків і запускаємо оркестратор повторно. Очікуємо статус `FAILED` для пошкодженого етапу. Ноутбуки, які стоять після нього, не повинні запускатися.

Після перевірки повертаємо змінений ноутбук у робочий стан.

У таблиці `nbu_meta.run_log` повинні залишитися записи обох прогонів, тому журнал завантажується через `WRITE_APPEND`.

#### Результати перевірки оркестратора

Під час першого запуску всі п’ять робочих ноутбуків виконалися успішно. Для всіх етапів був записаний однаковий `run_id`, а в таблиці `nbu_meta.run_log` з’явилися п’ять нових рядків.

Для перевірки зупинки конвеєра я тимчасово створив помилку в одному з робочих ноутбуків та запустив оркестратор повторно. Помилковий етап отримав статус `FAILED`, а наступні ноутбуки не запускалися.

Після перевірки я повернув змінений ноутбук у робочий стан.

Перевірочні записи з таблиці `nbu_meta.run_log`:

| `run_id` | `step_no` | `notebook` | `status` | `duration_sec` |
|---|---:|---|---|---:|
| `57225d1d-ca77-4065-83a6-c66278c28d81` | 3 | `03_dim_date.ipynb` | FAILED | 1.50 |
| `57225d1d-ca77-4065-83a6-c66278c28d81` | 2 | `02_dim_currency.ipynb` | OK | 6.96 |
| `57225d1d-ca77-4065-83a6-c66278c28d81` | 1 | `01_bronze.ipynb` | OK | 10.03 |
| `7bb036f3-299c-474b-93de-30324718ecb8` | 5 | `05_fact_merge.ipynb` | OK | 10.69 |
| `7bb036f3-299c-474b-93de-30324718ecb8` | 4 | `04_snapshot.ipynb` | OK | 8.46 |
| `7bb036f3-299c-474b-93de-30324718ecb8` | 3 | `03_dim_date.ipynb` | OK | 7.60 |
| `7bb036f3-299c-474b-93de-30324718ecb8` | 2 | `02_dim_currency.ipynb` | OK | 7.18 |
| `7bb036f3-299c-474b-93de-30324718ecb8` | 1 | `01_bronze.ipynb` | OK | 9.85 |